# <center>**Chapter 13 - Saving and Loading Model Weights**<center>

Theo ví dụ Iris trong chương: mạng 4 → 8 → 3 dùng ReLU, LogSoftmax và NLLLoss. Notebook xem state_dict, lưu trọng số riêng xuống đĩa, tạo lại kiến trúc rồi nạp weights để inference.

In [1]:
import csv
from pathlib import Path

import torch
from torch import nn
from torch.optim import Adam
from sklearn.model_selection import train_test_split

def find_file(filename):
    start = Path.cwd().resolve()
    for root in (start, *start.parents):
        for pattern in ("DLPytorch*.pdf", "*/DLPytorch*.pdf", "*/*/DLPytorch*.pdf"):
            for textbook in root.glob(pattern):
                candidate = textbook.parent / "data" / filename
                if candidate.is_file():
                    return candidate
    raise FileNotFoundError(f"Cannot find {filename} beside the local textbook PDF")

In [2]:
torch.manual_seed(42)
iris_path = find_file("iris.csv")
with iris_path.open(newline="", encoding="utf-8-sig") as file:
    rows = list(csv.DictReader(file))
feature_names = ("sepal_length", "sepal_width", "petal_length", "petal_width")
class_names = ("Iris-setosa", "Iris-versicolor", "Iris-virginica")
class_to_index = {name: index for index, name in enumerate(class_names)}
X = torch.tensor([[float(row[name]) for name in feature_names] for row in rows], dtype=torch.float32)
y = torch.tensor([class_to_index[row["species"]] for row in rows], dtype=torch.long)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, shuffle=True, random_state=42
)

In [3]:
class Multiclass(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(4, 8)
        self.activation = nn.ReLU()
        self.output = nn.Linear(8, 3)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, features):
        return self.log_softmax(self.output(self.activation(self.hidden(features))))

model = Multiclass()
loss_fn = nn.NLLLoss()
optimizer = Adam(model.parameters(), lr=0.01)
for epoch in range(500):
    model.train()
    loss = loss_fn(model(X_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        model.eval()
        with torch.no_grad():
            accuracy = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
        print(f"Epoch {epoch + 1:3d} | loss={loss.item():.4f} | test accuracy={accuracy:.1%}")

Epoch 100 | loss=0.4238 | test accuracy=97.8%
Epoch 200 | loss=0.1574 | test accuracy=97.8%
Epoch 300 | loss=0.0960 | test accuracy=97.8%
Epoch 400 | loss=0.0781 | test accuracy=100.0%
Epoch 500 | loss=0.0706 | test accuracy=100.0%


In [4]:
# state_dict chứa các tham số có thể lưu và nạp vào kiến trúc tương ứng.
print("state_dict keys:", list(model.state_dict()))
checkpoint = iris_path.parent / "iris-model.pth"
torch.save(model.state_dict(), checkpoint)

state_dict keys: ['hidden.weight', 'hidden.bias', 'output.weight', 'output.bias']


In [5]:
restored_model = Multiclass()
restored_model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
restored_model.eval()
with torch.no_grad():
    probabilities = restored_model(X_test)
    accuracy = (probabilities.argmax(dim=1) == y_test).float().mean().item()
print(f"Restored model test accuracy: {accuracy:.1%}")
print("Saved state_dict to:", checkpoint)

Restored model test accuracy: 100.0%
Saved state_dict to: C:\Users\qhoan\.vscode\Coding\Data Science\(!!!) 4.HK1 - Học sâu (Python)\Buổi thực hành số 6 (Multilayer Perceptron)\Lab06a - MLP Param Finetune\data\iris-model.pth


`state_dict` chứa tham số của mô hình đã huấn luyện. Lưu và nạp riêng state_dict giúp khôi phục trọng số mà không cần lưu toàn bộ định nghĩa lớp.

# <center>**HẾT**<center>